In this notebook, I make the metacell input by extracting data by sample from the concatenated, post-processed, SCVI, batch-corrected data. 

In [ ]:
pip install seacells

In [2]:
import sys
import os
import pandas as pd
import re
import numpy as np
import glob
from pathlib import Path
from scipy import sparse
from copy import deepcopy
import seaborn as sns
import csv
import itertools
import warnings
import scanpy as sc
import pickle as pkl
import SEACells
import gzip

In [3]:
# Path to the combined CSV file
combined_csv_path = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/postprocess_adata/postprocess_adata.013124/obs.combined.postprocessing.csv'

# Read the combined CSV file
combined_df = pd.read_csv(combined_csv_path, sep='\t')

In [4]:
combined_df

,Unnamed: 0,background_fraction,cell_probability,cell_size,droplet_efficiency,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,...,mito_frac,Patient,Tumor_Site,Culture_Media,ZFP_Expression,Replicate,Batch,Sample,phenograph,leiden
0,146P_HISC_shCTRL_CCGGACACACCACATA-1,0.022291,0.999955,14050.672852,1.324225,4761,8.468423,16009,9.680969,40.983197,...,0.150103,146,Primary,HISC,CTRL,1,6,146_P_HISC_CTRL_1,12,11
1,146P_dedifferentiation_shZFP36L2_3_ACTTTGTTCGC...,0.051227,0.999950,14127.892578,1.150659,3471,8.152486,12909,9.465757,42.389031,...,0.040127,146,Primary,Dedifferentiated,ZFP_KD,1,8,146_P_Dediff_ZFPKD_1,18,19
2,KG146Li_BASE_shZFP36L2_3_GTGGTTAGTTGGGAAC-1,0.007611,0.999955,14619.753906,2.005845,5775,8.661467,26207,10.173820,36.402488,...,0.059221,146,Metastatic,BASE,ZFP_KD,1,2,146_M_BASE_ZFPKD_1,13,13
3,146P_dedifferentiation_shCtrl_GGGTCTGTCGATGCTA-1,0.032237,0.999955,15622.842773,1.357936,4567,8.426831,17502,9.770128,39.389784,...,0.084676,146,Primary,Dedifferentiated,CTRL,1,8,146_P_Dediff_CTRL_1,10,10
4,146P_HISC_shZFP36L2_3_TGGAACTCAACCTAAC-1,0.013318,0.999955,15020.916016,1.453675,5019,8.521185,19114,9.858229,30.375641,...,0.065816,146,Primary,HISC,ZFP_KD,1,6,146_P_HISC_ZFPKD_1,0,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51135,146Li_dedifferentiation_shZFP36L2_4_AAATGGAAGC...,0.018784,0.999946,15664.033203,1.131694,3917,8.273337,15358,9.639457,46.627165,...,0.081847,146,Metastatic,Dedifferentiated,ZFP_KD,2,4,146_M_Dediff_ZFPKD_2,20,20
51136,146P_BASE_shZFP36L2_3_TCGTGGGAGCTCTTCC-1,0.056776,1.000000,8108.023438,0.866909,2184,7.689371,5582,8.627482,37.782157,...,0.059835,146,Primary,BASE,ZFP_KD,1,5,146_P_BASE_ZFPKD_1,3,5
51137,146Li_dedifferentiation_shCtrl_TAGACCAAGTAGACCG-1,0.022374,0.999881,14368.451172,1.182307,3695,8.215006,14463,9.579418,47.735601,...,0.072461,146,Metastatic,Dedifferentiated,CTRL,1,4,146_M_Dediff_CTRL_1,22,21
51138,146P_dedifferentiation_shCtrl_GGGCGTTAGAAGGCTC-1,0.092006,0.999999,11828.569336,0.860723,2583,7.857094,7451,8.916238,44.584620,...,0.103342,146,Primary,Dedifferentiated,CTRL,1,8,146_P_Dediff_CTRL_1,16,16


In [5]:
combined_df['Sample'].unique()
len(combined_df['Sample'].unique())

18

In [15]:
combined_df['Sample'].value_counts()

Sample
146_P_BASE_ZFPKD_1      4068
146_P_BASE_CTRL_1       3833
146_P_BASE_ZFPKD_2      3671
146_P_Dediff_ZFPKD_2    3661
146_P_HISC_ZFPKD_2      3559
146_P_Dediff_ZFPKD_1    3194
146_P_HISC_CTRL_1       3151
146_P_HISC_ZFPKD_1      3001
146_M_BASE_ZFPKD_1      2921
146_P_Dediff_CTRL_1     2809
146_M_HISC_ZFPKD_1      2795
146_M_HISC_CTRL_1       2588
146_M_BASE_ZFPKD_2      2580
146_M_BASE_CTRL_1       2436
146_M_Dediff_ZFPKD_2    2253
146_M_Dediff_ZFPKD_1    2133
146_M_Dediff_CTRL_1     2114
146_M_HISC_ZFPKD_2       373
Name: count, dtype: int64

In [17]:
# Iterate through each sample and extract the corresponding rows
# for sample in combined_df['Sample'].unique():
#     # Extract all rows and columns for the current sample
#     sample_df = combined_df[combined_df['Sample'] == sample]

#     sample_csv_path = f'/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/{sample}/obs.{sample}.csv'
#     sample_df.to_csv(sample_csv_path, sep='\t', index=False)
    

for sample in combined_df['Sample'].unique():
    sample_df = combined_df[combined_df['Sample'] == sample]

    # Construct the directory path
    sample_dir = f'/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/{sample}/'

    # Create the directory if it doesn't exist
    os.makedirs(sample_dir, exist_ok=True)

    # Construct the CSV file path
    sample_csv_path = f'{sample_dir}obs.{sample}.csv'

    # Save the DataFrame to CSV
    sample_df.to_csv(sample_csv_path, sep='\t', index=False)

In [6]:
sample_df

,Unnamed: 0,background_fraction,cell_probability,cell_size,droplet_efficiency,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,...,log10_original_total_counts,mito_frac,Patient,Tumor_Site,Culture_Media,ZFP_Expression,Replicate,Batch,Sample,phenograph
4498,Dedifferentiated_125P_shZFP36L2_3_CTGCTCAAGGAA...,0.047129,0.999874,50495.781250,2.099122,8366,9.032051,65568,11.090858,35.947413,...,4.816692,0.058336,125,Primary,Dedifferentiated,ZFP_KD,1,7,125_P_Dediff_ZFPKD_1,4
5062,Dedifferentiated_125P_shZFP36L2_3_ATGCATGAGAGC...,0.053267,0.999727,47274.542969,2.074584,8506,9.048645,58030,10.968733,37.332414,...,4.763653,0.098535,125,Primary,Dedifferentiated,ZFP_KD,1,7,125_P_Dediff_ZFPKD_1,4
10144,Dedifferentiated_125P_shZFP36L2_3_CCGATCTGTTTG...,0.090889,0.999893,43383.335938,1.429407,6766,8.819813,33388,10.415982,37.342758,...,4.523590,0.050138,125,Primary,Dedifferentiated,ZFP_KD,1,7,125_P_Dediff_ZFPKD_1,4
15798,Dedifferentiated_125P_shZFP36L2_3_TTGAACGAGCCA...,0.054784,0.999840,47052.574219,2.102878,7789,8.960596,58282,10.973066,45.516626,...,4.765534,0.051851,125,Primary,Dedifferentiated,ZFP_KD,1,7,125_P_Dediff_ZFPKD_1,4
18058,Dedifferentiated_125P_shZFP36L2_3_CTCAGAAAGGTT...,0.065672,0.999594,41812.765625,2.182142,7133,8.872627,49041,10.800432,45.054138,...,4.690559,0.066536,125,Primary,Dedifferentiated,ZFP_KD,1,7,125_P_Dediff_ZFPKD_1,4
18377,Dedifferentiated_125P_shZFP36L2_3_AACTTCTAGTAA...,0.065005,0.999610,42762.871094,2.053493,7556,8.930230,48026,10.779519,41.094407,...,4.681476,0.076354,125,Primary,Dedifferentiated,ZFP_KD,1,7,125_P_Dediff_ZFPKD_1,3
20251,Dedifferentiated_125P_shZFP36L2_3_TGAGACTGTTGA...,0.036519,0.999969,54421.070312,2.436097,9042,9.109746,86140,11.363741,45.596703,...,4.935205,0.086684,125,Primary,Dedifferentiated,ZFP_KD,1,7,125_P_Dediff_ZFPKD_1,3
20456,Dedifferentiated_125P_shZFP36L2_3_CATACAGAGCCT...,0.032372,0.999959,62884.566406,2.179239,9651,9.174920,95352,11.465341,32.205932,...,4.979330,0.062914,125,Primary,Dedifferentiated,ZFP_KD,1,7,125_P_Dediff_ZFPKD_1,4
22676,Dedifferentiated_125P_shZFP36L2_3_GGTCTGGCACTT...,0.049813,0.999848,50271.683594,1.979566,8459,9.043104,61250,11.022735,34.494694,...,4.787106,0.065339,125,Primary,Dedifferentiated,ZFP_KD,1,7,125_P_Dediff_ZFPKD_1,4
23857,Dedifferentiated_125P_shZFP36L2_3_GCCGATGGTTTG...,0.051301,0.999744,48321.050781,2.093401,8409,9.037177,60767,11.014819,38.384320,...,4.783668,0.053253,125,Primary,Dedifferentiated,ZFP_KD,1,7,125_P_Dediff_ZFPKD_1,4


In [19]:
# Path to the h5ad file
h5ad_file_path = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/postprocess_adata/postprocess_adata.013124/adata.combined.postprocessing.h5ad'

# Read the h5ad file
adata = sc.read_h5ad(h5ad_file_path)

In [17]:
adata

AnnData object with n_obs × n_vars = 2598 × 32485
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'log10GenesPerUMI', 'original_total_counts', 'log10_original_total_counts', 'mito_frac', 'Patient', 'Tumor_Site', 'Culture_Media', 'ZFP_Expression', 'Replicate', 'Batch', 'Sample', 'phenograph'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ribo', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'hvg', 'log1p', 'neighbors', 'num_components', 'paga', 'phenograph_sizes', 'umap', 'var_explained'
    obsm: 'X_pca', 'X_umap', 'gene_expression_encoding'
    layers: 'counts', 'without_log'
  

In [21]:
# # Iterate through each unique sample
# for sample in adata.obs['Sample'].unique():
#     # Extract data corresponding to the current sample
#     adata_sample = adata[adata.obs['Sample'] == sample].copy()

#     # Print the content of the extracted sample
# #     print(f"Sample: {sample}")
# #     print(adata_sample)
# #     print("=" * 50)

#     # Save the new h5ad file for each sample
#     output_path = f'/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells_noscvi/{sample}/adata.{sample}.h5ad'
#     adata_sample.write_h5ad(output_path)

# Iterate through each unique sample
for sample in adata.obs['Sample'].unique():
    # Extract data corresponding to the current sample
    adata_sample = adata[adata.obs['Sample'] == sample].copy()

    # Construct the directory path
    sample_dir = f'/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/{sample}/'

    # Create the directory if it doesn't exist
    os.makedirs(sample_dir, exist_ok=True)

    # Construct the h5ad file path
    output_path = f'{sample_dir}adata.{sample}.h5ad'

    # Save the new h5ad file for each sample
    adata_sample.write_h5ad(output_path)

## Trying out SEACells

In [20]:
# Read a sample h5ad file
ad = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells_noscvi/125_P_BASE_ZFPKD_1/adata.125_P_BASE_ZFPKD_1.h5ad')

In [41]:
ad

AnnData object with n_obs × n_vars = 2598 × 32485
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'log10GenesPerUMI', 'original_total_counts', 'log10_original_total_counts', 'mito_frac', 'Patient', 'Tumor_Site', 'Culture_Media', 'ZFP_Expression', 'Replicate', 'Batch', 'Sample', 'phenograph', 'SEACell'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ribo', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'hvg', 'log1p', 'neighbors', 'num_components', 'paga', 'phenograph_sizes', 'umap', 'var_explained'
    obsm: 'X_pca', 'X_umap', 'gene_expression_encoding'
    layers: 'counts', 'with

In [42]:
ad_mc = sc.AnnData(ad.raw.X.copy())

In [43]:
ad_mc

AnnData object with n_obs × n_vars = 2598 × 32485

In [44]:
ad_mc.obs_names = ad.raw.obs_names
ad_mc.obs = ad.obs
ad_mc.var_names = ad.raw.var_names
ad_mc.obsm = ad.obsm
ad_mc.uns = ad.uns

In [45]:
ad_mc

AnnData object with n_obs × n_vars = 2598 × 32485
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'log10GenesPerUMI', 'original_total_counts', 'log10_original_total_counts', 'mito_frac', 'Patient', 'Tumor_Site', 'Culture_Media', 'ZFP_Expression', 'Replicate', 'Batch', 'Sample', 'phenograph', 'SEACell'
    uns: 'hvg', 'log1p', 'neighbors', 'num_components', 'paga', 'phenograph_sizes', 'umap', 'var_explained'
    obsm: 'X_pca', 'X_umap', 'gene_expression_encoding'

In [31]:
build_kernel_on = 'X_pca'

In [35]:
## Additional parameters
n_waypoint_eigs = 10 # Number of eigenvalues to consider when initializing metacells
n_SEACells = 10

In [36]:
model = SEACells.core.SEACells(ad_mc, 
                  build_kernel_on=build_kernel_on, 
                  n_SEACells=n_SEACells, 
                  n_waypoint_eigs=n_waypoint_eigs,
                  convergence_epsilon = 1e-5)

Welcome to SEACells!


In [37]:
model.construct_kernel_matrix()
M = model.kernel_matrix

Computing kNN graph using scanpy NN ...
Computing radius for adaptive bandwidth kernel...


  0%|          | 0/2598 [00:00<?, ?it/s]

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/2598 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/2598 [00:00<?, ?it/s]

Constructing CSR matrix...


In [46]:
# Initialize archetypes
model.initialize_archetypes()

model.fit(min_iter=10, max_iter=200)

SEACell_ad = SEACells.core.summarize_by_SEACell(ad_mc, SEACells_label='SEACell', summarize_layer='X')


Building kernel on X_pca
Computing diffusion components from X_pca for waypoint initialization ... 
Done.
Sampling waypoints ...
Done.
Selecting 9 cells from waypoint initialization.
Initializing residual matrix using greedy column selection
Initializing f and g...


100%|██████████| 11/11 [00:00<00:00, 406.67it/s]

Selecting 1 cells from greedy initialization.
Randomly initialized A matrix.


Starting iteration 1.
Completed iteration 1.
Starting iteration 10.
Completed iteration 10.
Converged after 17 iterations.


100%|██████████| 10/10 [00:00<00:00, 29.74it/s]
/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


In [ ]:
adata.obs['Sample'].unique()

In [10]:
ad_mc = sc.AnnData(ad.raw.X.copy())
# ad_mc.obs_names = ad.raw.obs_names
# ad_mc.obs = ad.obs
# ad_mc.var_names = ad.raw.var_names
# ad_mc.obsm = ad.obsm
# ad_mc.uns = ad.uns

In [11]:
ad_mc

AnnData object with n_obs × n_vars = 2247 × 32491

In [7]:
ad_mc.raw = ad_mc

In [8]:
ad_mc

AnnData object with n_obs × n_vars = 2247 × 32491
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'log10GenesPerUMI', 'original_total_counts', 'log10_original_total_counts', 'mito_frac', 'Patient', 'Tumor_Site', 'Culture_Media', 'ZFP_Expression', 'Replicate', 'Batch', 'Sample', 'phenograph', 'leiden'
    uns: 'diffmap_evals', 'leiden', 'log1p', 'neighbors', 'num_components', 'paga', 'phenograph_sizes', 'rank_genes_groups', 'umap', 'var_explained'
    obsm: 'X_diffmap', 'X_pca', 'X_umap', 'gene_expression_encoding'

In [201]:
# Check if the raw attribute is present
if ad.raw is not None and ad.raw.X is not None:
    # Make a copy of ad.raw.X
    ad_mc = sc.AnnData(ad.raw.X.copy())
    print("Copy of ad.raw.X created.")
else:
    print("No valid raw attribute found in AnnData object.")

Copy of ad.raw.X created.


In [203]:
ad_mc

AnnData object with n_obs × n_vars = 2247 × 32491

In [207]:
ad.raw.X.shape

(2247, 32491)

In [209]:
build_kernel_on = 'X_pca'

In [210]:
n_waypoint_eigs = 10

In [211]:
model = SEACells.core.SEACells(ad_mc, 
                  build_kernel_on=build_kernel_on, 
                  n_SEACells=n_SEACells, 
                  n_waypoint_eigs=n_waypoint_eigs,
                  convergence_epsilon = 1e-5)

model.construct_kernel_matrix()
M = model.kernel_matrix

Welcome to SEACells!
Computing kNN graph using scanpy NN ...
Computing radius for adaptive bandwidth kernel...


  0%|          | 0/2247 [00:00<?, ?it/s]

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/2247 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/2247 [00:00<?, ?it/s]

Constructing CSR matrix...


In [212]:
model.initialize_archetypes()

model.fit(min_iter=10, max_iter=200)

Building kernel on X_pca
Computing diffusion components from X_pca for waypoint initialization ... 
Done.
Sampling waypoints ...
Done.
Selecting 9 cells from waypoint initialization.
Initializing residual matrix using greedy column selection
Initializing f and g...


100%|██████████| 11/11 [00:00<00:00, 387.77it/s]

Selecting 1 cells from greedy initialization.
Randomly initialized A matrix.
Setting convergence threshold at 0.00089
Starting iteration 1.


Completed iteration 1.
Starting iteration 10.
Completed iteration 10.
Starting iteration 20.
Completed iteration 20.
Starting iteration 30.
Completed iteration 30.
Converged after 31 iterations.


In [213]:
model.get_hard_assignments()['SEACell'].unique()

array(['SEACell-0', 'SEACell-3', 'SEACell-7', 'SEACell-8', 'SEACell-6',
       'SEACell-9', 'SEACell-4', 'SEACell-1', 'SEACell-2', 'SEACell-5'],
      dtype=object)

In [ ]:
def summarize_by_SEACell(
    ad, SEACells_label="SEACell", celltype_label=None, summarize_layer="raw"
):
    """Summary of SEACell assignment.

    Aggregates cells within each SEACell, summing over all raw data for all cells belonging to a SEACell.
    Data is unnormalized and raw aggregated counts are stored .layers['raw'].
    Attributes associated with variables (.var) are copied over, but relevant per SEACell attributes must be
    manually copied, since certain attributes may need to be summed, or averaged etc, depending on the attribute.
    The output of this function is an anndata object of shape n_metacells x original_data_dimension.
    :return: anndata.AnnData containing aggregated counts.

    """
    import scanpy as sc
    from scipy.sparse import csr_matrix

    # Set of metacells
    metacells = ad.obs[SEACells_label].unique()

    # Summary matrix
    summ_matrix = pd.DataFrame(0.0, index=metacells, columns=ad.var_names)

    for m in tqdm(summ_matrix.index):
        cells = ad.obs_names[ad.obs[SEACells_label] == m]
        if summarize_layer == "X":
            summ_matrix.loc[m, :] = np.ravel(ad[cells, :].X.sum(axis=0))
        elif summarize_layer == "raw" and ad.raw is not None:
            summ_matrix.loc[m, :] = np.ravel(ad[cells, :].raw.X.sum(axis=0))
        else:
            summ_matrix.loc[m, :] = np.ravel(
                ad[cells, :].layers[summarize_layer].sum(axis=0)
            )

    # Ann data

    # Counts
    meta_ad = sc.AnnData(csr_matrix(summ_matrix), dtype=csr_matrix(summ_matrix).dtype)
    meta_ad.obs_names, meta_ad.var_names = summ_matrix.index.astype(str), ad.var_names
    meta_ad.layers["raw"] = csr_matrix(summ_matrix)

    # Also compute cell type purity
    if celltype_label is not None:
        # TODO: Catch specific exception
        try:
            purity_df = evaluate.compute_celltype_purity(ad, celltype_label)
            meta_ad.obs = meta_ad.obs.join(purity_df)
        except Exception as e:  # noqa: BLE001
            print(f"Cell type purity failed with Exception {e}")

    return meta_ad

In [214]:
SEACell_ad = SEACells.core.summarize_by_SEACell(ad_mc, SEACells_label='SEACell', summarize_layer='raw')

100%|██████████| 10/10 [00:00<00:00, 74.62it/s]
/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


In [32]:
print(sample)

0        125_P_BASE_CTRL_1
1       125_P_BASE_ZFPKD_1
2       125_P_BASE_ZFPKD_2
3        125_P_HISC_CTRL_1
4       125_P_HISC_ZFPKD_1
5       125_P_HISC_ZFPKD_2
6        146_P_BASE_CTRL_1
7       146_P_BASE_ZFPKD_1
8       146_P_BASE_ZFPKD_2
9        146_P_HISC_CTRL_1
10      146_P_HISC_ZFPKD_1
11      146_P_HISC_ZFPKD_2
12     146_P_Dediff_CTRL_1
13    146_P_Dediff_ZFPKD_1
14    146_P_Dediff_ZFPKD_2
15     125_P_Dediff_CTRL_1
16    125_P_Dediff_ZFPKD_1
17    125_P_Dediff_ZFPKD_2
18       146_M_BASE_CTRL_1
19      146_M_BASE_ZFPKD_1
20      146_M_BASE_ZFPKD_2
21       146_M_HISC_CTRL_1
22      146_M_HISC_ZFPKD_1
23      146_M_HISC_ZFPKD_2
24     146_M_Dediff_CTRL_1
25    146_M_Dediff_ZFPKD_1
26    146_M_Dediff_ZFPKD_2
Name: Sample, dtype: object


In [34]:
# Read the sample information from the CSV file
aliases = pd.read_csv('/data/chanjlab/forsythb/organoid_analysis_pipeline_scripts/Organoid_Sample_Description.txt', sep='\t', header=0)

# Iterate through each sample in the DataFrame
for index, row in aliases.iterrows():
    sample = row['Sample']
    
    # Rest of your code
    #n_SEACells = int(sys.argv[2])
    mode = 'combined'
    comb_dir = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/out_postproc_scvi/scvi.latent_100/'
    ind_dir = f'/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/{sample}/'
    out_dir = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output/metacells/'
    os.makedirs(out_dir, exist_ok=True)

    ind_file = ind_dir + f'adata.{sample}.h5ad'
    #ad = sc.read_h5ad(ind_file)
    print(ind_file)


/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/125_P_BASE_CTRL_1/adata.125_P_BASE_CTRL_1.h5ad
/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/125_P_BASE_ZFPKD_1/adata.125_P_BASE_ZFPKD_1.h5ad
/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/125_P_BASE_ZFPKD_2/adata.125_P_BASE_ZFPKD_2.h5ad
/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/125_P_HISC_CTRL_1/adata.125_P_HISC_CTRL_1.h5ad
/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/125_P_HISC_ZFPKD_1/adata.125_P_HISC_ZFPKD_1.h5ad
/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/125_P_HISC_ZFPKD_2/adata.125_P_HISC_ZFPKD_2.h5ad
/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/146_P_BASE_CTRL_1/adata.146_P_BASE_CTRL_1.h5ad
/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/146_P_BASE_ZFPKD_1/adata.146_P_BASE_ZFPKD_1.h5ad
/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/146_P_BASE_ZFPKD_2/adata.146_P_BASE_ZFPKD_2.h5ad
/data/chanjlab/CRC_ZFP36L2.092023/

In [44]:
ad = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/125_P_BASE_CTRL_1/adata.125_P_BASE_CTRL_1.h5ad')

In [39]:
build_kernel_on = 'X_pca' # key in ad.obsm to use for computing metacells
                          # This would be replaced by 'X_svd' for ATAC data

## Additional parameters
n_waypoint_eigs = 10 # Number of eigenvalues to consider when initializing metacells

n_SEACells = 10

model = SEACells.core.SEACells(ad, 
                  build_kernel_on=build_kernel_on, 
                  n_SEACells=n_SEACells, 
                  n_waypoint_eigs=n_waypoint_eigs,
                  convergence_epsilon = 1e-5)

Welcome to SEACells!


In [54]:
model.construct_kernel_matrix()
M = model.kernel_matrix
print(M)
print(M.shape)

Computing kNN graph using scanpy NN ...
Computing radius for adaptive bandwidth kernel...


  0%|          | 0/2247 [00:00<?, ?it/s]

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/2247 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/2247 [00:00<?, ?it/s]

Constructing CSR matrix...
  (0, 0)	1.0
  (0, 29)	0.35386728291843456
  (0, 232)	0.3047898432532922
  (0, 315)	0.3025327487504726
  (0, 333)	0.401589847964942
  (0, 420)	0.2623632722816272
  (0, 718)	0.3217101223564541
  (0, 746)	0.37346506041152566
  (0, 747)	0.32320081055078126
  (0, 765)	0.3109513007747515
  (0, 809)	0.348631408804292
  (0, 909)	0.34171603647097526
  (0, 912)	0.32741613792645685
  (0, 936)	0.3556135331596556
  (0, 1018)	0.24718077910877345
  (0, 1058)	0.29306381375672497
  (0, 1100)	0.33751197204811845
  (0, 1145)	0.3579517688506569
  (0, 1173)	0.3233889322035006
  (0, 1231)	0.30768864960710424
  (0, 1233)	0.38691207433233193
  (0, 1387)	0.35119506435490166
  (0, 1400)	0.3309093015065278
  (0, 1535)	0.3384224625467047
  (0, 1641)	0.3668712506399082
  :	:
  (2246, 329)	0.2832645890965139
  (2246, 370)	0.40452369427670304
  (2246, 553)	0.32694634408970286
  (2246, 557)	0.2317652478935101
  (2246, 578)	0.2881123027489399
  (2246, 598)	0.2548734057857721
  (2246, 649)	0

In [55]:
# Initialize archetypes
model.initialize_archetypes()

model.fit(min_iter=10, max_iter=200)

# def custom_summarize_by_SEACell(ad, SEACells_label='SEACell', summarize_layer='raw'):
#     try:
#         SEACell_ad = SEACells.core.summarize_by_SEACell(ad, SEACells_label=SEACells_label, summarize_layer=summarize_layer)
#         return SEACell_ad
#     except ValueError as e:
#         print(f"Error summarizing SEACells for sample {sample}: {e}")
#         return None

# SEACell_ad = custom_summarize_by_SEACell(ad, SEACells_label='SEACell', summarize_layer='raw')

# Check if SEACell_ad is None before proceeding
# if SEACell_ad is not None:
#     odir=ind_dir + f'SEACells/{n_SEACells}/'
#     os.makedirs(odir, exist_ok=True)
#     ofile = odir + f'SEACell_adata.{sample}.{n_SEACells}.h5ad'
#     SEACell_ad.write_h5ad(ofile)

#     model.save_assignments(odir)

#     model.get_hard_assignments().to_csv(odir + 'mc_assignments.csv')


Building kernel on X_pca
Computing diffusion components from X_pca for waypoint initialization ... 
Done.
Sampling waypoints ...
Done.
Selecting 9 cells from waypoint initialization.
Initializing residual matrix using greedy column selection
Initializing f and g...


100%|██████████| 11/11 [00:00<00:00, 365.18it/s]

Selecting 1 cells from greedy initialization.
Randomly initialized A matrix.
Starting iteration 1.


Completed iteration 1.
Starting iteration 10.
Completed iteration 10.
Starting iteration 20.
Completed iteration 20.
Converged after 20 iterations.


In [60]:
SEACell_ad = SEACells.core.summarize_by_SEACell(ad, SEACells_label='SEACell', summarize_layer='raw')

KeyError: 'SEACell'

In [62]:
ad.obs['SEACells_label']

KeyError: 'SEACells_label'

In [43]:
# Initialize archetypes
model.initialize_archetypes()

model.fit(min_iter=10, max_iter=200)

SEACell_ad = SEACells.core.summarize_by_SEACell(ad, SEACells_label='SEACell', summarize_layer='raw')


Building kernel on X_pca
Computing diffusion components from X_pca for waypoint initialization ... 
Done.
Sampling waypoints ...
Done.
Selecting 9 cells from waypoint initialization.
Initializing residual matrix using greedy column selection
Initializing f and g...


100%|██████████| 11/11 [00:00<00:00, 376.76it/s]

Selecting 1 cells from greedy initialization.
Randomly initialized A matrix.
Starting iteration 1.


Completed iteration 1.
Starting iteration 10.
Completed iteration 10.
Starting iteration 20.
Completed iteration 20.
Converged after 20 iterations.


  0%|          | 0/10 [00:00<?, ?it/s]


ValueError: could not broadcast input array from shape (32491,) into shape (5000,)

In [50]:
ad.raw

In [ ]:
ad_mc.raw[:, 'orig_variable_name'].X

In [75]:
ad = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/metacells/125_P_BASE_CTRL_1/adata.125_P_BASE_CTRL_1.h5ad')

In [76]:
ad

AnnData object with n_obs × n_vars = 2247 × 5000
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'log10GenesPerUMI', 'original_total_counts', 'log10_original_total_counts', 'mito_frac', 'Patient', 'Tumor_Site', 'Culture_Media', 'ZFP_Expression', 'Replicate', 'Batch', 'Sample', '_scvi_batch', '_scvi_labels', 'phenograph', 'leiden'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ribo', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'diffmap_evals', 'hvg', 'leiden', 'log1p', 'neighbors', 'num_components', 'paga', 'phenograph_sizes', 'rank_genes_gro

In [77]:
ad.obsm['X_pca']

array([[ 1.01721186e+01,  9.45849548e+00, -4.04634814e+00, ...,
        -9.83361863e-01, -1.77192505e-01,  7.34899803e-01],
       [ 7.59420284e+00,  8.39125970e+00, -4.79510821e+00, ...,
         8.61184199e-02, -1.18639418e+00, -4.96426538e-01],
       [ 1.23982948e+01, -7.17099131e-01, -5.05994491e+00, ...,
         1.74778317e+00,  2.99515401e+00, -8.95267651e-01],
       ...,
       [ 1.06508527e+01,  8.13678407e+00, -4.03848417e+00, ...,
        -4.80578368e-02,  6.88388334e-03,  5.09377878e-01],
       [ 8.91238158e+00,  1.14809361e+01, -4.44274782e+00, ...,
        -1.10269218e+00,  1.82000085e-01,  8.40757764e-02],
       [ 1.02161368e+01,  1.10049930e+01, -4.26007877e+00, ...,
        -7.45852291e-01, -1.99267061e-01,  1.54991055e+00]])

In [68]:
build_kernel_on = 'X_pca' # key in ad.obsm to use for computing metacells

In [69]:
n_waypoint_eigs = 10

In [70]:
model = SEACells.core.SEACells(ad, 
                  build_kernel_on=build_kernel_on, 
                  n_SEACells=n_SEACells, 
                  n_waypoint_eigs=n_waypoint_eigs,
                  convergence_epsilon = 1e-5)

Welcome to SEACells!


In [74]:
ad.obsm['X_pca']

array([[ 1.01721186e+01,  9.45849548e+00, -4.04634814e+00, ...,
        -9.83361863e-01, -1.77192505e-01,  7.34899803e-01],
       [ 7.59420284e+00,  8.39125970e+00, -4.79510821e+00, ...,
         8.61184199e-02, -1.18639418e+00, -4.96426538e-01],
       [ 1.23982948e+01, -7.17099131e-01, -5.05994491e+00, ...,
         1.74778317e+00,  2.99515401e+00, -8.95267651e-01],
       ...,
       [ 1.06508527e+01,  8.13678407e+00, -4.03848417e+00, ...,
        -4.80578368e-02,  6.88388334e-03,  5.09377878e-01],
       [ 8.91238158e+00,  1.14809361e+01, -4.44274782e+00, ...,
        -1.10269218e+00,  1.82000085e-01,  8.40757764e-02],
       [ 1.02161368e+01,  1.10049930e+01, -4.26007877e+00, ...,
        -7.45852291e-01, -1.99267061e-01,  1.54991055e+00]])

In [78]:
model.construct_kernel_matrix()
M = model.kernel_matrix

Computing kNN graph using scanpy NN ...
Computing radius for adaptive bandwidth kernel...


  0%|          | 0/2247 [00:00<?, ?it/s]

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/2247 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/2247 [00:00<?, ?it/s]

Constructing CSR matrix...


In [80]:
print(M.shape)

(2247, 2247)


In [82]:
ad.obsm['X_pca']

array([[ 1.01721186e+01,  9.45849548e+00, -4.04634814e+00, ...,
        -9.83361863e-01, -1.77192505e-01,  7.34899803e-01],
       [ 7.59420284e+00,  8.39125970e+00, -4.79510821e+00, ...,
         8.61184199e-02, -1.18639418e+00, -4.96426538e-01],
       [ 1.23982948e+01, -7.17099131e-01, -5.05994491e+00, ...,
         1.74778317e+00,  2.99515401e+00, -8.95267651e-01],
       ...,
       [ 1.06508527e+01,  8.13678407e+00, -4.03848417e+00, ...,
        -4.80578368e-02,  6.88388334e-03,  5.09377878e-01],
       [ 8.91238158e+00,  1.14809361e+01, -4.44274782e+00, ...,
        -1.10269218e+00,  1.82000085e-01,  8.40757764e-02],
       [ 1.02161368e+01,  1.10049930e+01, -4.26007877e+00, ...,
        -7.45852291e-01, -1.99267061e-01,  1.54991055e+00]])

In [89]:
model.get_hard_assignments()


,SEACell
index,
125P_BASE_shCTRL_TTCGATTGTGTTTCTT-1,SEACell-7
125P_BASE_shCTRL_TCGCTCAAGATGAAGG-1,SEACell-7
125P_BASE_shCTRL_TCCTGCATCCATTTCA-1,SEACell-8
125P_BASE_shCTRL_TCTTTGATCGGCACTG-1,SEACell-6
125P_BASE_shCTRL_ACAGGGAAGCATTTGC-1,SEACell-1
...,...
125P_BASE_shCTRL_GAAGTAAAGTCGAAAT-1,SEACell-9
125P_BASE_shCTRL_GAGCCTGCAGCCTACG-1,SEACell-7
125P_BASE_shCTRL_GTAATCGAGTTCACTG-1,SEACell-7


In [103]:
def summarize_by_SEACell(
    ad, SEACells_label="SEACell", celltype_label=None, summarize_layer="raw"
):
    """Summary of SEACell assignment.

    Aggregates cells within each SEACell, summing over all raw data for all cells belonging to a SEACell.
    Data is unnormalized and raw aggregated counts are stored .layers['raw'].
    Attributes associated with variables (.var) are copied over, but relevant per SEACell attributes must be
    manually copied, since certain attributes may need to be summed, or averaged etc, depending on the attribute.
    The output of this function is an anndata object of shape n_metacells x original_data_dimension.
    :return: anndata.AnnData containing aggregated counts.

    """
    import scanpy as sc
    from scipy.sparse import csr_matrix

    # Set of metacells
    metacells = ad.obs[SEACells_label].unique()
    print(metacells)

    # Summary matrix
    summ_matrix = pd.DataFrame(0.0, index=metacells, columns=ad.var_names)
    print(summ_matrix)
        

    for m in tqdm(summ_matrix.index):
#         cells = ad.obs_names[ad.obs[SEACells_label] == m]

#         if summarize_layer == "X":
#             result = ad[cells, :].X.sum(axis=0)
#         elif summarize_layer == "raw" and ad.raw is not None:
#             result = ad[cells, :].raw.X.sum(axis=0)
#         else:
#             result = ad[cells, :].layers[summarize_layer].sum(axis=0)

#         print(f"Shape of result: {result.shape}")
#         print(f"Shape of summ_matrix.loc[{m}, :]: {summ_matrix.loc[m, :].shape}")

#         summ_matrix.loc[m, :] = np.ravel(result)

#     # Ann data

#     # Counts
#     meta_ad = sc.AnnData(csr_matrix(summ_matrix), dtype=csr_matrix(summ_matrix).dtype)
#     meta_ad.obs_names, meta_ad.var_names = summ_matrix.index.astype(str), ad.var_names
#     meta_ad.layers["raw"] = csr_matrix(summ_matrix)

#     # Also compute cell type purity
#     if celltype_label is not None:
#         # TODO: Catch specific exception
#         try:
#             purity_df = evaluate.compute_celltype_purity(ad, celltype_label)
#             meta_ad.obs = meta_ad.obs.join(purity_df)
#         except Exception as e:  # noqa: BLE001
#             print(f"Cell type purity failed with Exception {e}")

#     return meta_ad


In [104]:
# Initialize archetypes
model.initialize_archetypes()

model.fit(min_iter=10, max_iter=200)

SEACell_ad = summarize_by_SEACell(ad, SEACells_label='SEACell', summarize_layer='raw')

Building kernel on X_pca
Computing diffusion components from X_pca for waypoint initialization ... 
Done.
Sampling waypoints ...
Done.
Selecting 9 cells from waypoint initialization.
Initializing residual matrix using greedy column selection
Initializing f and g...


100%|██████████| 11/11 [00:00<00:00, 370.16it/s]

Selecting 1 cells from greedy initialization.
Randomly initialized A matrix.
Starting iteration 1.


Completed iteration 1.
Starting iteration 10.
Completed iteration 10.
Starting iteration 20.
Completed iteration 20.
Converged after 20 iterations.
['SEACell-7' 'SEACell-8' 'SEACell-6' 'SEACell-1' 'SEACell-2' 'SEACell-4'
 'SEACell-3' 'SEACell-5' 'SEACell-9' 'SEACell-0']
gene_name  A2ML1  AADACL2  AADACL2-AS1  AADAT  AARD  ABAT  ABCA1  ABCA13  \
SEACell-7    0.0      0.0          0.0    0.0   0.0   0.0    0.0     0.0   
SEACell-8    0.0      0.0          0.0    0.0   0.0   0.0    0.0     0.0   
SEACell-6    0.0      0.0          0.0    0.0   0.0   0.0    0.0     0.0   
SEACell-1    0.0      0.0          0.0    0.0   0.0   0.0    0.0     0.0   
SEACell-2    0.0      0.0          0.0    0.0   0.0   0.0    0.0     0.0   
SEACell-4    0.0      0.0          0.0    0.0   0.0   0.0    0.0     0.0   
SEACell-3    0.0      0.0          0.0    0.0   0.0   0.0    0.0     0.0   
SEACell-5    0.0      0.0          0.0    0.0   0.0   0.0    0.0     0.0   
SEACell-9    0.0      0.0          0.0    0.0

In [84]:
# Assuming 'model' is an instance of your SEACells model
labels = model.get_hard_assignments()
ad.obs["SEACell"] = labels["SEACell"]

In [85]:
print(labels)

                                       SEACell
index                                         
125P_BASE_shCTRL_TTCGATTGTGTTTCTT-1  SEACell-7
125P_BASE_shCTRL_TCGCTCAAGATGAAGG-1  SEACell-7
125P_BASE_shCTRL_TCCTGCATCCATTTCA-1  SEACell-8
125P_BASE_shCTRL_TCTTTGATCGGCACTG-1  SEACell-6
125P_BASE_shCTRL_ACAGGGAAGCATTTGC-1  SEACell-1
...                                        ...
125P_BASE_shCTRL_GAAGTAAAGTCGAAAT-1  SEACell-9
125P_BASE_shCTRL_GAGCCTGCAGCCTACG-1  SEACell-7
125P_BASE_shCTRL_GTAATCGAGTTCACTG-1  SEACell-7
125P_BASE_shCTRL_CTGAATGTCAAACGTC-1  SEACell-3
125P_BASE_shCTRL_TCGACGGTCCGCGGAT-1  SEACell-3

[2247 rows x 1 columns]


In [90]:
ad.obs["SEACell"] = labels["SEACell"]

In [91]:
ad

AnnData object with n_obs × n_vars = 2247 × 5000
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'log10GenesPerUMI', 'original_total_counts', 'log10_original_total_counts', 'mito_frac', 'Patient', 'Tumor_Site', 'Culture_Media', 'ZFP_Expression', 'Replicate', 'Batch', 'Sample', '_scvi_batch', '_scvi_labels', 'phenograph', 'leiden', 'SEACell'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ribo', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'diffmap_evals', 'hvg', 'leiden', 'log1p', 'neighbors', 'num_components', 'paga', 'phenograph_sizes', 'ran

In [93]:
print(np.ravel(ad[cells, :].raw.X.sum(axis=0)).shape)
print(summ_matrix.loc[m, :].shape)

NameError: name 'cells' is not defined